In [1]:
import pandas as pd
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import numpy as np

# 1. Load the Dataset
try:
    df = pd.read_csv('underwriter_dataset_1000.csv')
except FileNotFoundError:
    print("Error: 'underwriter_dataset_1000.csv' not found.")
    exit()

# 2. Preprocess the Data
X = df.drop(['Risk_Level_Assessed', 'Name', 'Underwriter_ID'], axis=1)
y = df['Risk_Level_Assessed']

# One-hot encode categorical columns
X = pd.get_dummies(X, columns=['Insurance_Type', 'State', 'Certifications'], drop_first=True)

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Train the XGBoost Model
print("Training the XGBoost model...")
model = xgb.XGBClassifier(
    objective='multi:softprob',
    n_estimators=100,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='mlogloss'
)
model.fit(X_train, y_train)
print("Model training complete.")
print("-" * 30)

# 5. Evaluate the Model
print("Evaluating the XGBoost model...")
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
# determine which encoded labels actually appear in true/pred (avoid size mismatch)
labels_present = np.unique(np.concatenate([y_test, y_pred]))

# replace any NaN class name with 'Medium'
def _clean_label(v):
    if pd.isna(v) or str(v).lower() == "nan":
        return "Medium"
    return str(v)

target_names_present = [_clean_label(le.classes_[int(l)]) for l in labels_present]
print(classification_report(y_test, y_pred, labels=labels_present, target_names=target_names_present))
print("-" * 30)

# 6. EXPLAIN the Model's Predictions with SHAP
print("Generating SHAP explanations...")
explainer = shap.TreeExplainer(model)
# Generate shap values
shap_values = explainer.shap_values(X_test)

print("SHAP values type:", type(shap_values))

# --- CASE 1: If list (one array per class) ---
if isinstance(shap_values, list):
    for i, class_name in enumerate(le.classes_):
        plt.figure()
        shap.summary_plot(shap_values[i], X_test, show=False)
        plt.title(f"SHAP Summary Plot - Class: {class_name}")
        plt.savefig(f'shap_summary_class_{class_name}.png', bbox_inches='tight')
        plt.close()
        print(f"--> SHAP Summary Plot saved for class '{class_name}'")

# --- CASE 2: If numpy array (3D: samples x features x classes) ---
else:
    for i, class_name in enumerate(le.classes_):
        plt.figure()
        shap.summary_plot(shap_values[:, :, i], X_test, show=False)
        plt.title(f"SHAP Summary Plot - Class: {class_name}")
        plt.savefig(f'shap_summary_class_{class_name}.png', bbox_inches='tight')
        plt.close()
        print(f"--> SHAP Summary Plot saved for class '{class_name}'")

# Now plot safely
# Choose the instance index you want to explain (0..len(X_test)-1)
instance_index = 0  # change to the example you want to inspect

# Use the earlier predictions for class selection (y_pred was computed above)
predicted_class = int(y_pred[instance_index])

# Derive base_value from the explainer (handles list or scalar/array)
base_val = explainer.expected_value
if isinstance(base_val, list) or isinstance(base_val, np.ndarray):
    try:
        base_value = float(np.array(base_val)[predicted_class].squeeze())
    except Exception:
        # fallback: take first scalar available
        base_value = float(np.array(base_val).ravel()[0])
else:
    base_value = float(base_val)

# Extract the shap values for the chosen instance and class
if isinstance(shap_values, list):
    # shap_values is a list with one array per class (n_samples x n_features)
    sv_instance = shap_values[predicted_class][instance_index]
else:
    # shap_values is a 3D array (n_samples x n_features x n_classes)
    sv_instance = shap_values[instance_index, :, predicted_class]

# Create and save the force plot (matplotlib renderer)
force_plot = shap.force_plot(
    base_value,
    sv_instance,
    X_test.iloc[instance_index, :],
    matplotlib=True,
    show=False
)
plt.savefig('shap_force_plot.png', bbox_inches='tight')
plt.close()
print("--> SHAP Force Plot saved as 'shap_force_plot.png'")


# --- SHAP Summary Plots ---
print("\nGenerating SHAP Summary Plots...")

# Normalize shap_values into a list of (n_samples x n_features) arrays, one per class
def to_class_list(shap_vals, X):
    arr = np.asarray(shap_vals)
    n_samples, n_features = X.shape[0], X.shape[1]

    # Case: list already provided
    if isinstance(shap_vals, list):
        class_list = [np.asarray(sv) for sv in shap_vals]
    else:
        # ndarray cases
        if arr.ndim == 3:
            # try samples x features x classes
            if arr.shape[0] == n_samples and arr.shape[1] >= 1:
                # last axis is classes
                class_list = [arr[:, :, i] for i in range(arr.shape[2])]
            # try classes x samples x features
            elif arr.shape[0] <= 50 and arr.shape[1] == n_samples and arr.shape[2] == n_features:
                class_list = [arr[i] for i in range(arr.shape[0])]
            # try samples x (features+1) x classes (rare)
            elif arr.shape[0] == n_samples and arr.shape[2] == n_features:
                class_list = [arr[:, :, i] for i in range(arr.shape[2])]
            else:
                raise AssertionError(f"Unexpected shap_values shape: {arr.shape}")
        elif arr.ndim == 2:
            # single-class style -> treat as one class
            class_list = [arr]
        else:
            raise AssertionError(f"Unexpected shap_values ndim: {arr.ndim}")

    # Ensure each class matrix matches X shape (trim trailing constant column if present)
    normalized = []
    for sv in class_list:
        sv = np.asarray(sv)
        if sv.ndim != 2:
            raise AssertionError(f"Each class SHAP must be 2D, got shape {sv.shape}")
        if sv.shape[1] == n_features:
            normalized.append(sv)
        elif sv.shape[1] == n_features + 1:
            # trim last column (common when an extra constant offset column is present)
            normalized.append(sv[:, :n_features])
        else:
            raise AssertionError(
                f"The shape of a class shap_values matrix ({sv.shape}) does not match X_test ({n_samples}, {n_features})."
            )
    return normalized

try:
    class_shap_list = to_class_list(shap_values, X_test)
except AssertionError as e:
    print("Error normalizing shap_values:", e)
    raise

# (A) Class-specific summary
for i, class_name in enumerate(le.classes_):
    sv = class_shap_list[i]
    plt.figure()
    shap.summary_plot(sv, X_test, show=False)
    plt.title(f"SHAP Summary Plot - Class: {class_name}")
    plt.savefig(f'shap_summary_class_{class_name}.png', bbox_inches='tight')
    plt.close()
    print(f"--> SHAP Summary Plot saved for class '{class_name}'")

# (B) Overall importance (mean across classes)
shap_values_mean = np.mean([np.abs(sv) for sv in class_shap_list], axis=0)
plt.figure()
shap.summary_plot(shap_values_mean, X_test, plot_type="bar", show=False)
plt.title("Overall Feature Importance (mean across classes)")
plt.savefig('shap_summary_mean.png', bbox_inches='tight')
plt.close()
print("--> Overall SHAP Summary Plot saved as 'shap_summary_mean.png'")

print("All SHAP plots generated successfully.")


Training the XGBoost model...
Model training complete.
------------------------------
Evaluating the XGBoost model...
Model Accuracy: 0.9500

Classification Report:
              precision    recall  f1-score   support

        High       0.00      0.00      0.00         1
         Low       0.50      1.00      0.67         1
      Medium       1.00      1.00      1.00        18

    accuracy                           0.95        20
   macro avg       0.50      0.67      0.56        20
weighted avg       0.93      0.95      0.93        20

------------------------------
Generating SHAP explanations...
SHAP values type: <class 'numpy.ndarray'>


c:\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [00:35:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0

--> SHAP Summary Plot saved for class 'High'
--> SHAP Summary Plot saved for class 'Low'
--> SHAP Summary Plot saved for class 'Medium'
--> SHAP Summary Plot saved for class 'nan'


c:\Python312\Lib\site-packages\shap\plots\_force_matplotlib.py:107: RuntimeWarning: divide by zero encountered in scalar divide
  feature_contribution = np.abs(float(feature[0]) - pre_val) / np.abs(total_effect)


--> SHAP Force Plot saved as 'shap_force_plot.png'

Generating SHAP Summary Plots...
--> SHAP Summary Plot saved for class 'High'
--> SHAP Summary Plot saved for class 'Low'
--> SHAP Summary Plot saved for class 'Medium'
--> SHAP Summary Plot saved for class 'nan'
--> Overall SHAP Summary Plot saved as 'shap_summary_mean.png'
All SHAP plots generated successfully.


Issue: classification_report fails because target_names contains non-string values (floats). Convert them to strings before calling classification_report.

Change the notebook where the classification report is printed.



In [2]:
# ...existing code...
print("\nClassification Report:")
# determine which encoded labels actually appear in true/pred (avoid size mismatch)
labels_present = np.unique(np.concatenate([y_test, y_pred]))

# replace any NaN class name with 'Medium'
def _clean_label(v):
    if pd.isna(v) or str(v).lower() == "nan":
        return "Medium"
    return str(v)

target_names_present = [_clean_label(le.classes_[int(l)]) for l in labels_present]
print(classification_report(y_test, y_pred, labels=labels_present, target_names=target_names_present))
print("-" * 30)
# ...existing code...


Classification Report:
              precision    recall  f1-score   support

        High       0.00      0.00      0.00         1
         Low       0.50      1.00      0.67         1
      Medium       1.00      1.00      1.00        18

    accuracy                           0.95        20
   macro avg       0.50      0.67      0.56        20
weighted avg       0.93      0.95      0.93        20

------------------------------


c:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))




This will fix the TypeError.

In [3]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Ensure predictions exist
y_pred = model.predict(X_test)

# 1) Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.ylabel('True')
plt.xlabel('Predicted')
plt.title('Confusion Matrix (Test)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.close()
print("--> Saved: confusion_matrix.png")

# Prepare top features using SHAP mean importance (recompute if needed)
try:
    shap_values_mean  # already computed earlier in notebook
except NameError:
    class_shap_list = to_class_list(shap_values, X_test)
    shap_values_mean = np.mean([np.abs(sv) for sv in class_shap_list], axis=0)

# Per-feature importance (mean across samples)
feat_imp = np.mean(shap_values_mean, axis=0)
top_k = 5
top_idx = np.argsort(feat_imp)[::-1][:top_k]
top_feats = list(X_test.columns[top_idx])

# 2) Violin plots for top features grouped by true class
plt.figure(figsize=(5 * len(top_feats), 4))
for i, f in enumerate(top_feats):
    plt.subplot(1, len(top_feats), i + 1)
    sns.violinplot(x=le.inverse_transform(y_test), y=X_test[f], order=le.classes_, cut=0)
    plt.title(f)
    plt.xlabel('')
plt.suptitle('Top feature distributions by true class')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('top_feature_violins.png', bbox_inches='tight')
plt.close()
print(f"--> Saved: top_feature_violins.png (features: {top_feats})")

# 3) Scatter of top-2 features colored by predicted class (if at least 2 features available)
if len(top_feats) >= 2:
    plt.figure(figsize=(6,5))
    pred_names = le.inverse_transform(y_pred)
    sns.scatterplot(x=X_test[top_feats[0]], y=X_test[top_feats[1]],
                    hue=pred_names, palette='tab10', alpha=0.75, edgecolor=None)
    plt.xlabel(top_feats[0])
    plt.ylabel(top_feats[1])
    plt.title('Top-2 features colored by predicted class')
    plt.legend(title='Predicted', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('top2_scatter_pred.png', bbox_inches='tight')
    plt.close()
    print("--> Saved: top2_scatter_pred.png")

# Extra: True vs Predicted distribution bar chart (concise summary)
plt.figure(figsize=(6,4))
true_counts = pd.Series(le.inverse_transform(y_test)).value_counts().reindex(le.classes_).fillna(0)
pred_counts = pd.Series(le.inverse_transform(y_pred)).value_counts().reindex(le.classes_).fillna(0)
df_counts = pd.DataFrame({'True': true_counts, 'Predicted': pred_counts})
df_counts.plot(kind='bar', rot=45)
plt.title('True vs Predicted class distribution (test)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('true_vs_pred_distribution.png', bbox_inches='tight')
plt.close()
print("--> Saved: true_vs_pred_distribution.png")

--> Saved: confusion_matrix.png
--> Saved: top_feature_violins.png (features: ['Policy_Decisions_Monthly', 'Years_Experience', 'Certifications_Fellow ANZIIF', 'Certifications_CPA', 'State_WA'])
--> Saved: top2_scatter_pred.png
--> Saved: true_vs_pred_distribution.png
